# Cropping and merging feeds

`fetch` crops the feeds it delivers to the area or place you ask for. The
functions it uses also work on their own, on any GTFS feed: `crop_feed` cuts a
feed to an area or a date window, `patch_feed` heals broken trips from a
sibling feed, and `merge_feeds` writes several feeds into one. This page also
shows how to crop the data of a fetch you already made to another area.

## Crop a feed

Let's take Turku's feed from the download cache and crop it to the city
centre, given as a bounding box:

In [1]:
import transitio

rec = transitio.place("Turku").recommend("2026-10-14")
result = transitio.fetch(feeds=rec, osm=False, progress=False)
path = result.feeds[0]

crop = transitio.crop_feed(path, "turku-centre.zip", aoi=(22.24, 60.44, 22.29, 60.46))
before = transitio.validate_feed(path)["row_counts"]
for name in ("stops.txt", "routes.txt", "trips.txt"):
    print(name, before[name], "->", crop["row_counts"][name])

stops.txt 2954 -> 2716
routes.txt 99 -> 77
trips.txt 15687 -> 13992


Notice that most stops stay. Trips serving at least one stop inside the area are kept with their full stop
sequences, or, with `full_trips_only=True`, only trips entirely inside;
`start_date` and `end_date` keep the trips whose service can run in that
window. Everything the kept trips do not use — stops, routes, shapes,
calendars and the rest — goes, so the cropped feed stays consistent. A
polygon crops to the polygon itself; a bounding box, a point or a line
crops to its bounding box.

A date window, given as `YYYYMMDD` strings, keeps one week of service:

In [2]:
week = transitio.crop_feed(path, "turku-week.zip", start_date="20261012", end_date="20261018")
week["row_counts"]["trips.txt"]

9932

## Crop after fetching

Say you fetched the data recommended for Turku, and your study area turns out
to be only the city centre. The simplest way to crop everything again is to
call `fetch` once more, with the new area and the recommendation's feeds and
day. The feeds come from the download cache, and the OpenStreetMap data is
cropped from an extract downloaded earlier when one covers the area:

In [3]:
centre = (22.24, 60.44, 22.29, 60.46)
centre_result = transitio.fetch(centre, when=rec.when, feeds=rec.feed_ids)
centre_result.selection_table()[["feed_id", "cache"]]

Fetching 1 feed for the area


[1/1] f-u6x-turunlinja~autoilijainosakeyhtiö~savonlinjaoy~sl~autolinja: cached copy reused


Fetching the OSM extract


Done in 2 s: 1 feed delivered, 0 skipped


,feed_id,cache
0,f-u6x-turunlinja~autoilijainosakeyhtiö~savonli...,reused


Notice that the `cache` column says `reused`: the feed was not downloaded
again.
The recommendation itself cannot go with an area, because it names its
place; its `feed_ids` and `when` can.

The other way crops the files of the first result yourself: `crop_feed` for
each feed and `fetch_pbf` for the OpenStreetMap data:

In [4]:
for n, feed_path in enumerate(result.feeds):
    transitio.crop_feed(feed_path, f"centre-{n}.zip", aoi=centre)
pbf = transitio.fetch_pbf(centre)
pbf.name

'bbox_22.24_60.44_22.29_60.46.osm.pbf'

**Be careful**: the feeds of `result` are already cropped to Turku, so this
works only for an area inside Turku; trips outside the first area are gone.
Calling `fetch` again has no such limit, and it also validates the cropped
feeds and checks the day, as the first call did.

## Patch a feed

```python
transitio.patch_feed(path, "sibling.zip", "patched.zip")  # heal broken trips
```

`patch_feed` repairs the broken trips of a feed with those of a sibling feed
that runs the same service.

## Merge feeds

`merge_feeds` writes one feed from several, such as the two feeds recommended
for Tallinn:

In [5]:
tallinn = transitio.place("Tallinn")
tallinn_result = transitio.fetch(feeds=tallinn.recommend("2026-10-14"), osm=False, progress=False)
merged = transitio.merge_feeds(tallinn_result.feeds, "tallinn.gtfs.zip", check=False)
merged["row_counts"]

{'agency.txt': 22,
 'calendar.txt': 894,
 'calendar_dates.txt': 23907,
 'fare_attributes.txt': 19,
 'routes.txt': 406,
 'shapes.txt': 568661,
 'stop_times.txt': 848761,
 'stops.txt': 4638,
 'transfers.txt': 700,
 'trips.txt': 33941}

`merge_feeds` writes one feed from the cropped ones. With `check=False` it
keeps the file when the validator reports ERROR notices; the returned
report lists them.